# 14 · Xuất artifacts cho app (ticket #20)

Gói những gì app cần để chạy không mạng, không torch: bảng bài (có tên bài, nghệ sĩ), embedding bài + `track_index` của mô hình gu tốt hơn giữa ALS và item2vec (NDCG@10 val cao nhất trong `{model}_params.json`), decay, `mix_params.json` (trọng số và α; mặc định cho đến khi #25 có kết quả), 100 playlist test mẫu và trọng số điểm mix học (#23).

**Input** (Add Input): `titeohihi/mixnext-splited`, `mixnext-embeddings` (ALS), `mixnext-item2vec`, `mixnext-mix-learned`.

**Output** (`/kaggle/working/artifacts/`): `tracks.parquet`, `item_embeddings.npy`, `track_index.parquet`, `demo_playlists.parquet`, `mix_params.json`, `mix_learned.npz`, `manifest.json`. Tạo Dataset `mixnext-artifacts` từ output (chỉ giữ `artifacts/`), rồi tải về và giải nén vào `artifacts/` của repo để chạy app.

Bật **Internet** để clone repo. Chỉ cần CPU.

In [ ]:
# Bootstrap: lấy code từ GitHub rồi cài package src/ (pyproject.toml, #3).
# Đang làm trên nhánh a/b/c thì đổi BRANCH; khi ticket đã merge thì để "main".
BRANCH = "c"
REPO = "/kaggle/working/MusicRecSys"
!rm -rf {REPO} && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys {REPO}
!pip install -q --no-deps -e {REPO}   # cho các lệnh `python -m src…` chạy ở tiến trình mới, từ thư mục bất kỳ
import sys; sys.path.insert(0, REPO)   # kernel đang chạy chỉ đọc gói mới cài khi khởi động lại: dòng này mới làm `import src` chạy được

In [ ]:
from src.export_artifacts import main

OUT = "/kaggle/working/artifacts"
main(["--input", "/kaggle/input", "--out", OUT])   # tìm file theo tên dưới /kaggle/input; báo lỗi nếu thiếu hoặc trùng

In [ ]:
# Kiểm tra như app: đọc lại chỉ bằng numpy/pandas, gợi ý cho một playlist demo, không import torch.
import subprocess
check = '''
import sys, time
from src.export_artifacts import load_artifacts
a = load_artifacts("/kaggle/working/artifacts")
seed = list(a.demo["seed_track_ids"].iloc[0])
t0 = time.perf_counter()
ids, _ = a.taste_model().candidates(seed)
print(f"top-100 cho một phiên: {time.perf_counter() - t0:.3f} giây")
learned = a.learned_mix()
mix = learned if learned is not None else a.rule_mix()
print("điểm mix 5 ứng viên đầu:", mix(seed[-1], ids)[:5].round(3))
print(a.tracks.set_index("track_id").loc[ids[:5], ["track_name", "artist_name", "bpm", "camelot"]])
print("torch đã import:", "torch" in sys.modules)
'''
print(subprocess.run([sys.executable, "-c", check], cwd=REPO, capture_output=True, text=True).stdout)
!du -sh {OUT}

## Kết quả (chạy trên Kaggle, CPU, ngày …)

Điền sau khi chạy: mô hình được chọn và NDCG@10 val của từng mô hình, số bài có vector, số bài bị cắt, dung lượng, thời gian top-100 cho một phiên.